## LOINC OCL ETL Output File Splitter

This notebook provides Python scripts to split large JSON output files generated by the LOINC OCL ETL process into smaller, more manageable chunks. This is particularly useful for uploading data to systems that have file size or record count limitations, such as OpenConceptLab (OCL).

Three different splitting methods are demonstrated:
1.  **Splitting by Maximum File Size (for line-delimited JSON):** Ideal for `concepts.json` and similar files where each line is a self-contained JSON object.
2.  **Splitting by Number of Lines (for line-delimited JSON):** Suitable for files like `mappings.json` where you need a consistent number of records per output file.
3.  **Splitting JSON Dictionary by Number of Lists or Maximum File Size:** Designed for files like `hierarchy_only.json` which contain a single JSON object where keys map to lists of data.

## Configuration

Before running the scripts, please review and adjust the configuration variables in each code block to match your specific file paths, desired output settings, and substitution requirements.

### General Configuration Parameters:
*   `inputfile`: The full path to the large input JSON file you want to split.
*   `folder`: The directory where the output chunk files will be saved. The script will create this folder if it does not exist.
*   `out_file_name_root`: The base name for the output chunk files. An index or other identifier will be appended to this name.

In [ ]:
# --- Configuration ---
inputfile = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-LOINC-2.81-22Sep25-Prod/concepts.json"
folder = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-LOINC-2.81-22Sep25-Prod/Split Concepts"
out_file_name_root = "concepts_chunks_22Sep25_"
max_file_size_mb = 30  # Maximum file size in MB

# --- Substitution ---
# Define the text to find and what to replace it with.
# If 'text_to_find' is an empty string (""), no substitution will occur.
text_to_find = "en-GB"
replacement_text = "en"


# --- File Splitting Logic ---
max_file_size_bytes = max_file_size_mb * 1024 * 1024
current_output_file_index = 1
current_output_file = None
current_output_file_size = 0

# Create the output folder if it doesn't exist
import json
import ndjson
import os
os.makedirs(folder, exist_ok=True)

Splitting 'concepts.json' into chunks of 30 MB...
Substituting all instances of 'en-GB' with 'en'.
Creating new file: concepts_chunks_22Sep25_1.json
Creating new file: concepts_chunks_22Sep25_2.json
Creating new file: concepts_chunks_22Sep25_3.json
Creating new file: concepts_chunks_22Sep25_4.json
Creating new file: concepts_chunks_22Sep25_5.json
Creating new file: concepts_chunks_22Sep25_6.json
Creating new file: concepts_chunks_22Sep25_7.json
Creating new file: concepts_chunks_22Sep25_8.json
Creating new file: concepts_chunks_22Sep25_9.json
Creating new file: concepts_chunks_22Sep25_10.json
Creating new file: concepts_chunks_22Sep25_11.json
Creating new file: concepts_chunks_22Sep25_12.json
Creating new file: concepts_chunks_22Sep25_13.json
Creating new file: concepts_chunks_22Sep25_14.json
Creating new file: concepts_chunks_22Sep25_15.json
Creating new file: concepts_chunks_22Sep25_16.json
Creating new file: concepts_chunks_22Sep25_17.json
Creating new file: concepts_chunks_22Sep25_

### 1. Splitting `concepts.json` (by File Size with Substitution)

This section splits a line-delimited JSON file (`concepts.json`) into chunks based on a maximum file size. It also includes an optional text substitution feature.

**Specific Configuration:**
*   `max_file_size_mb`: The maximum desired size for each output file in megabytes.
*   `text_to_find`: (Optional) A string to find within each line.
*   `replacement_text`: (Optional) The string to replace `text_to_find` with. If `text_to_find` is empty, no substitution will occur.

**To Use:**
1.  **Modify `inputfile`, `folder`, `out_file_name_root`, `max_file_size_mb`, `text_to_find`, and `replacement_text`** in the first code cell.
2.  **Run the code cell** to execute the splitting process.

In [ ]:
#Split file by maximum size
print(f"Splitting '{os.path.basename(inputfile)}' into chunks of {max_file_size_mb} MB...")
if text_to_find:
    print(f"Substituting all instances of '{text_to_find}' with '{replacement_text}'.")


with open(inputfile, 'r', encoding='utf8') as bigfile:
    for line in bigfile:

        # Perform substitution on the line if text_to_find is specified
        if text_to_find:
            modified_line = line.replace(text_to_find, replacement_text)
        else:
            modified_line = line

        line_size_bytes = len(modified_line.encode('utf-8'))


        # Check if a new file needs to be created based on the size of the MODIFIED line
        if current_output_file is None or current_output_file_size + line_size_bytes > max_file_size_bytes:
            if current_output_file:
                current_output_file.close()

            output_filename = f"{out_file_name_root}{current_output_file_index}.json"
            output_filepath = os.path.join(folder, output_filename)
            current_output_file = open(output_filepath, 'w', encoding='utf8')
            current_output_file_size = 0
            current_output_file_index += 1
            print(f"Creating new file: {output_filename}")

        # Write the modified line to the current file
        current_output_file.write(modified_line)
        current_output_file_size += line_size_bytes

if current_output_file:
    current_output_file.close()

print("File splitting complete! ✅")

### 2. Splitting `mappings.json` (by Number of Lines)

This section splits a line-delimited JSON file (`mappings.json`) into chunks, ensuring each output file contains a specified number of lines (records).

**Specific Configuration:**
*   `lines_per_file`: The exact number of lines (records) desired in each output chunk file.

**To Use:**
1.  **Modify `inputfile`, `folder`, `out_file_name_root`, and `lines_per_file`** in the second code cell.
2.  **(Optional) Uncomment and adjust the `line = line.replace(...)`** if you need to perform text substitution for this file.
3.  **Run the code cell** to execute the splitting process.

In [ ]:
# Split file by number of lines
inputfile = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-LOINC-2.81-22Sep25-Prod/mappings.json"
folder = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-LOINC-2.81-22Sep25-Prod/Split Mappings/"
out_file_name_root = "mappings_chunks_22Sep2025_"

lines_per_file=50000


lineno=0
bigJSON = open(inputfile, 'r', encoding='utf8')
smallfile = open(folder + out_file_name_root + str(lines_per_file) + ".json", "w", encoding='utf8')

with bigJSON as bigfile:
    for line in bigfile:

        #replace text as needed
        # line = line.replace('LOINC-3','LOINC')

        lineno+=1
        if lineno % lines_per_file == 0:
            if smallfile:
                smallfile.close()
            small_filename = (folder+(out_file_name_root + '{}'.format(lineno + lines_per_file))+'.json')
            smallfile = open(small_filename, "w", encoding='utf8')
        smallfile.write(line)
    if smallfile:
        smallfile.close()

### 3. Splitting `hierarchy_only.json` (JSON Dictionary)

This section handles JSON files that represent a single dictionary, where values are lists. It offers two splitting options for these types of files: by number of lists or by maximum file size.

**Specific Configuration:**
*   `split_by_lists`: Set to `True` to split by a fixed number of lists per file. Set to `False` to split by maximum file size.
*   `lists_per_file`: If `split_by_lists` is `True`, this specifies how many top-level lists (dictionary entries) go into each output file.
*   `split_by_size`: Set to `True` to split by a maximum file size. Set to `False` to split by number of lists.
*   `max_file_size_mb`: If `split_by_size` is `True`, this specifies the maximum desired size for each output file in megabytes.

**To Use:**
1.  **Modify `inputfile`, `folder`, `out_file_name_root`** in the third code cell.
2.  **Choose your splitting method:**
    *   Set `split_by_lists = True` and `split_by_size = False` to split by number of lists.
    *   Set `split_by_lists = False` and `split_by_size = True` to split by maximum file size.
3.  **Adjust `lists_per_file` or `max_file_size_mb`** based on your chosen method.
4.  **Run the code cell** to execute the splitting process.

In [ ]:
## Split file by lists in a JSON dictionary

# --- Configuration for hierarchy_only.json splitting ---
inputfile = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-LOINC-2.81-22Sep25-Prod/hierarchy_only.json"
folder    = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-LOINC-2.81-22Sep25-Prod/Split Hierarchy-5mb/"
# inputfile = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-arc/hierarchy_only.json"
# folder = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-arc/"

out_file_name_root = "hierarchy_chunks_22Sep2025_"


# Choose one splitting method and comment out the other
split_by_lists = False
lists_per_file = 2 # Change this to the desired number of lists per file

split_by_size = True
max_file_size_mb = 5 # Change this to the desired maximum file size in MB

# --- File Splitting Logic for JSON Dictionaries ---

# Load the entire JSON file into a Python dictionary
with open(os.path.join(folder, inputfile), 'r', encoding='utf8') as f:
    data = json.load(f)

# Get the list of items to process
items = list(data.items())

if split_by_lists:
    print(f"Splitting '{inputfile}' by number of lists, {lists_per_file} lists per file...")
    num_files = (len(items) + lists_per_file - 1) // lists_per_file
    for i in range(num_files):
        start_index = i * lists_per_file
        end_index = min((i + 1) * lists_per_file, len(items))
        chunk = items[start_index:end_index]

        output_filename = f"{out_file_name_root}by_list_{i+1}.json"
        output_filepath = os.path.join(folder, output_filename)

        with open(output_filepath, 'w', encoding='utf8') as outfile:
            json.dump(dict(chunk), outfile, indent=2)

        print(f"Created file: {output_filename}")

elif split_by_size:
    print(f"Splitting '{inputfile}' by maximum file size, {max_file_size_mb} MB per file...")
    max_file_size_bytes = max_file_size_mb * 1024 * 1024

    current_output_file_index = 1
    current_output_data = {}

    for key, value in items:
        # Check size before adding the next item
        temp_data = current_output_data.copy()
        temp_data[key] = value

        temp_size = len(json.dumps(temp_data, indent=2).encode('utf-8'))

        if temp_size > max_file_size_bytes and current_output_data:
            # Write the current file
            output_filename = f"{out_file_name_root}by_size_{current_output_file_index}.json"
            output_filepath = os.path.join(folder, output_filename)
            with open(output_filepath, 'w', encoding='utf8') as outfile:
                json.dump(current_output_data, outfile, indent=2)
            print(f"Created file: {output_filename}")

            # Start a new file with the current item
            current_output_file_index += 1
            current_output_data = {key: value}
        else:
            current_output_data[key] = value

    # Write any remaining data to the last file
    if current_output_data:
        output_filename = f"{out_file_name_root}by_size_{current_output_file_index}.json"
        output_filepath = os.path.join(folder, output_filename)
        with open(output_filepath, 'w', encoding='utf8') as outfile:
            json.dump(current_output_data, outfile, indent=2)
        print(f"Created file: {output_filename}")

print("File splitting complete! ✅")

Splitting 'C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-LOINC-2.81-22Sep25-Prod/hierarchy_only.json' by maximum file size, 5 MB per file...
Created file: hierarchy_chunks_22Sep2025_by_size_1.json
Created file: hierarchy_chunks_22Sep2025_by_size_2.json
Created file: hierarchy_chunks_22Sep2025_by_size_3.json
Created file: hierarchy_chunks_22Sep2025_by_size_4.json
File splitting complete! ✅
